#### Imports and Library Configs
##### Core Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
import random

In [ ]:
import warnings

warnings.filterwarnings("ignore")

##### TensorFlow and Keras imports

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, DenseNet201, ResNet50, EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Set TensorFlow random seed
tf.random.set_seed(42)

##### Scikit-learn metrics

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

##### Random Seeds and Plot Config

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)

# Set style for professional plots
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
sns.set_context("notebook", font_scale=1.2)

#### Reduced Dataset

In [ ]:
def create_reduced_dataset(
    original_dir="dataset", target_dir="dataset_small", samples_per_class=200
):
    """
    Create a smaller balanced dataset for fast training.
    This helps reduce training time while maintaining class balance.

    Args:
        original_dir: Path to original dataset
        target_dir: Path where reduced dataset will be saved
        samples_per_class: Number of samples to take per class for training

    Returns:
        target_dir: Path to the reduced dataset
    """

    # Check if reduced dataset already exists
    if os.path.exists(target_dir):
        print(f"📁 Reduced dataset already exists at {target_dir}")
        return target_dir

    print(f"🔄 Creating reduced dataset with {samples_per_class} samples per class...")
    print("-" * 50)

    for split in ["Training", "Testing"]:
        for class_name in ["Normal", "Stroke"]:
            source_path = os.path.join(original_dir, split, class_name)
            target_path = os.path.join(target_dir, split, class_name)

            if not os.path.exists(source_path):
                print(f"⚠️ Warning: {source_path} not found")
                continue

            os.makedirs(target_path, exist_ok=True)

            # Get all image files
            images = [
                f
                for f in os.listdir(source_path)
                if f.lower().endswith((".png", ".jpg", ".jpeg"))
            ]

            # Select appropriate number of samples
            if split == "Training":
                selected = random.sample(images, min(samples_per_class, len(images)))
            else:
                # Use all testing images or limit to 100
                selected = random.sample(images, min(100, len(images)))

            # Copy selected images to target directory
            for img in selected:
                shutil.copy2(
                    os.path.join(source_path, img), os.path.join(target_path, img)
                )

            print(f"   📂 {split}/{class_name}: {len(selected)} images")

    print(f"✅ Reduced dataset created at {target_dir}")
    return target_dir


# Create the reduced dataset
base_dir = create_reduced_dataset(samples_per_class=200)

#### Data Augmentation

In [ ]:
def load_data(base_dir="dataset_small", img_size=(224, 224), batch_size=16):
    """
    Load and prepare the dataset with train/validation/test splits.

    Args:
        base_dir: Path to dataset directory
        img_size: Target image size (height, width)
        batch_size: Batch size for training

    Returns:
        train_generator, validation_generator, test_generator
    """

    train_dir = os.path.join(base_dir, "Training")
    test_dir = os.path.join(base_dir, "Testing")

    # Verify directories exist
    if not os.path.exists(train_dir):
        raise FileNotFoundError(f"Training directory not found: {train_dir}")
    if not os.path.exists(test_dir):
        raise FileNotFoundError(f"Testing directory not found: {test_dir}")

    # Data augmentation for training (helps prevent overfitting)
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,  # Normalize pixel values to [0,1]
        rotation_range=10,  # Small rotation (medical images don't need large rotations)
        horizontal_flip=True,  # Horizontal flip (symmetry in brain scans)
        validation_split=0.2,  # Reserve 20% for validation
    )

    # No augmentation for testing (only rescaling)
    test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

    # Training generator (80% of training data)
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="binary",
        subset="training",
        seed=42,
        shuffle=True,
    )

    # Validation generator (20% of training data)
    validation_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="binary",
        subset="validation",
        seed=42,
        shuffle=True,
    )

    # Test generator (no shuffling for consistent evaluation)
    test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="binary",
        shuffle=False,
        seed=42,
    )

    # Print dataset statistics
    print("\n" + "=" * 60)
    print("📊 DATASET STATISTICS")
    print("=" * 60)
    print(f"Training samples:   {train_generator.samples}")
    print(f"Validation samples: {validation_generator.samples}")
    print(f"Testing samples:    {test_generator.samples}")
    print(f"Classes:            {list(train_generator.class_indices.keys())}")
    print("=" * 60)

    return train_generator, validation_generator, test_generator


# Load the data
train_generator, validation_generator, test_generator = load_data()

#### Dataset Overview

In [ ]:
def plot_dataset_stats(train_gen, valid_gen, test_gen):

    # Plot 1-3: Dataset sizes across splits
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    datasets = ["Training", "Validation", "Testing"]
    samples = [train_gen.samples, valid_gen.samples, test_gen.samples]
    colors = ["#2E86AB", "#A23B72", "#F18F01"]

    for i, (ax, name, count) in enumerate(zip(axes, datasets, samples)):
        ax.bar(name, count, color=colors[i], edgecolor="black", linewidth=1.5)
        ax.set_ylabel("Number of Images", fontsize=11)
        ax.set_title(f"{name} Set\n{count} samples", fontsize=12, fontweight="bold")
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle(
        "Dataset Distribution by Split", fontsize=14, fontweight="bold", y=1.05
    )
    plt.tight_layout()
    plt.show()

    # Plot 4: Class distribution
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    class_names = list(train_gen.class_indices.keys())
    train_counts = []
    test_counts = []

    # Count images per class
    train_dir = train_gen.directory
    test_dir = test_gen.directory

    for class_name in class_names:
        train_count = len(
            [
                f
                for f in os.listdir(os.path.join(train_dir, class_name))
                if f.lower().endswith((".png", ".jpg", ".jpeg"))
            ]
        )
        test_count = len(
            [
                f
                for f in os.listdir(os.path.join(test_dir, class_name))
                if f.lower().endswith((".png", ".jpg", ".jpeg"))
            ]
        )
        train_counts.append(train_count)
        test_counts.append(test_count)

    x = np.arange(len(class_names))
    width = 0.35

    ax.bar(
        x - width / 2,
        train_counts,
        width,
        label="Training",
        color="#2E86AB",
        edgecolor="black",
        linewidth=1,
    )
    ax.bar(
        x + width / 2,
        test_counts,
        width,
        label="Testing",
        color="#F18F01",
        edgecolor="black",
        linewidth=1,
    )

    ax.set_xlabel("Class", fontsize=12)
    ax.set_ylabel("Number of Images", fontsize=12)
    ax.set_title("Class Distribution Across Datasets", fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(class_names)
    ax.legend(loc="upper right")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


# Visualize dataset statistics
plot_dataset_stats(train_generator, validation_generator, test_generator)

#### Model Building Function

In [ ]:
def build_model(
    base_model_name, input_shape=(224, 224, 3), dropout_rate=0.5, l2_reg=0.001
):
    """
    Build a transfer learning model with custom classification head.

    Args:
        base_model_name: Name of the pre-trained model to use
        input_shape: Input image dimensions
        dropout_rate: Dropout rate for regularization
        l2_reg: L2 regularization factor

    Returns:
        Compiled Keras model
    """

    # Available pre-trained models
    base_models = {
        "VGG16": VGG16,
        "DenseNet201": DenseNet201,
        "ResNet50": ResNet50,
        "EfficientNetB0": EfficientNetB0,
    }

    if base_model_name not in base_models:
        raise ValueError(
            f"Base model {base_model_name} not supported. "
            f"Choose from {list(base_models.keys())}"
        )

    # Load pre-trained base model (without top classification layer)
    base_model = base_models[base_model_name](
        weights="imagenet", include_top=False, input_shape=input_shape
    )
    base_model.trainable = False  # Freeze base model weights

    # Build custom classification head
    inputs = tf.keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)  # Reduce spatial dimensions
    x = Dropout(dropout_rate)(x)  # Prevent overfitting
    outputs = Dense(1, activation="sigmoid")(x)  # Binary classification

    model = Model(inputs, outputs)

    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )

    return model

#### Feature Extraction and Training

In [ ]:
# Extract features using pre-trained model, then train only dense layers
#


def extract_features_and_train(base_model_name, train_gen, valid_gen, test_gen):
    """
    - Feature extraction using pre-trained model
    - Only train dense layers
    - Significantly faster than training the full model, while still leveraging powerful feature extraction.

    Args:
        base_model_name: Name of the pre-trained model to use
        train_gen: Training data generator
        valid_gen: Validation data generator
        test_gen: Testing data generator

    Returns:
        dense_model: Trained classification model
        history: Training history
        y_test: True labels
        y_pred: Predicted labels
        y_pred_prob: Predicted probabilities
    """

    print(f"\n{'=' * 60}")
    print(f"🚀 Processing {base_model_name} - Feature Extraction Approach")
    print(f"{'=' * 60}\n")

    # Load base model
    base_models = {
        "VGG16": VGG16,
        "DenseNet201": DenseNet201,
        "ResNet50": ResNet50,
        "EfficientNetB0": EfficientNetB0,
    }

    base_model = base_models[base_model_name](
        weights="imagenet", include_top=False, input_shape=(224, 224, 3)
    )

    # Extract features from training data
    print("📤 Extracting training features...")
    train_features = []
    train_labels = []
    for i in range(len(train_gen)):
        x, y = train_gen[i]
        features = base_model.predict(x, verbose=0)
        train_features.append(features)
        train_labels.append(y)

    X_train = np.vstack(train_features)
    y_train = np.hstack(train_labels)

    # Extract features from validation data
    print("📤 Extracting validation features...")
    valid_features = []
    valid_labels = []
    for i in range(len(valid_gen)):
        x, y = valid_gen[i]
        features = base_model.predict(x, verbose=0)
        valid_features.append(features)
        valid_labels.append(y)

    X_valid = np.vstack(valid_features)
    y_valid = np.hstack(valid_labels)

    # Extract features from test data
    print("📤 Extracting test features...")
    test_features = []
    test_labels = []
    for i in range(len(test_gen)):
        x, y = test_gen[i]
        features = base_model.predict(x, verbose=0)
        test_features.append(features)
        test_labels.append(y)

    X_test = np.vstack(test_features)
    y_test = np.hstack(test_labels)

    print("\n✅ Feature extraction complete!")
    print(f"   Training features shape: {X_train.shape}")
    print(f"   Validation features shape: {X_valid.shape}")
    print(f"   Test features shape: {X_test.shape}")

    # Build small dense model on top of extracted features
    print("\n🏗️ Building classification head...")
    feature_shape = X_train.shape[1:]

    dense_model = tf.keras.Sequential(
        [
            tf.keras.layers.GlobalAveragePooling2D(input_shape=feature_shape),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )

    dense_model.compile(
        optimizer=Adam(learning_rate=0.001),  # Higher LR for dense layers
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    # Train only dense layers
    print("\n🎯 Training classification head...")
    history = dense_model.fit(
        X_train,
        y_train,
        validation_data=(X_valid, y_valid),
        epochs=20,
        batch_size=32,
        verbose=1,
        callbacks=[
            EarlyStopping(patience=5, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(patience=3, factor=0.5, verbose=0),
        ],
    )

    # Evaluate on test set
    test_loss, test_acc = dense_model.evaluate(X_test, y_test, verbose=0)
    print(f"\n📊 {base_model_name} Test Accuracy: {test_acc:.4f}")

    # Get predictions for visualization
    y_pred_prob = dense_model.predict(X_test, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()

    return dense_model, history, y_test, y_pred, y_pred_prob

#### Training History

In [ ]:
def plot_training_history(history, model_name):

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(history.history["accuracy"]) + 1)

    # Accuracy plot
    axes[0].plot(
        epochs,
        history.history["accuracy"],
        "o-",
        color="#2E86AB",
        linewidth=2,
        markersize=4,
        label="Training",
    )
    axes[0].plot(
        epochs,
        history.history["val_accuracy"],
        "o-",
        color="#F18F01",
        linewidth=2,
        markersize=4,
        label="Validation",
    )
    axes[0].set_xlabel("Epochs", fontsize=11)
    axes[0].set_ylabel("Accuracy", fontsize=11)
    axes[0].set_title(f"{model_name} - Accuracy", fontsize=12, fontweight="bold")
    axes[0].legend(loc="lower right")
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0, 1])

    # Loss plot
    axes[1].plot(
        epochs,
        history.history["loss"],
        "o-",
        color="#2E86AB",
        linewidth=2,
        markersize=4,
        label="Training",
    )
    axes[1].plot(
        epochs,
        history.history["val_loss"],
        "o-",
        color="#F18F01",
        linewidth=2,
        markersize=4,
        label="Validation",
    )
    axes[1].set_xlabel("Epochs", fontsize=11)
    axes[1].set_ylabel("Loss", fontsize=11)
    axes[1].set_title(f"{model_name} - Loss", fontsize=12, fontweight="bold")
    axes[1].legend(loc="upper right")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

#### Confusion Matrix (Single Model)

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name):

    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=True,
        xticklabels=["Normal", "Stroke"],
        yticklabels=["Normal", "Stroke"],
        square=True,
        linewidths=1,
        linecolor="black",
        annot_kws={"size": 14, "weight": "bold"},
    )

    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    ax.set_title(f"{model_name} - Confusion Matrix", fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.show()

#### ROC Curve (Single Model)

In [ ]:
def plot_roc_curve(y_true, y_prob, model_name):

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)

    fig, ax = plt.subplots(figsize=(7, 6))

    ax.plot(fpr, tpr, color="#2E86AB", lw=3, label=f"ROC Curve (AUC = {roc_auc:.3f})")
    ax.plot(
        [0, 1], [0, 1], color="#A23B72", lw=2, linestyle="--", label="Random Classifier"
    )
    ax.fill_between(fpr, tpr, alpha=0.2, color="#2E86AB")

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(f"{model_name} - ROC Curve", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

#### Precision-Recall Curve (Single Model)

In [ ]:
def plot_precision_recall_curve(y_true, y_prob, model_name):

    precision, recall, _ = precision_recall_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(7, 6))

    ax.plot(recall, precision, color="#F18F01", lw=3, label="Precision-Recall Curve")
    ax.set_xlabel("Recall", fontsize=12)
    ax.set_ylabel("Precision", fontsize=12)
    ax.set_title(
        f"{model_name} - Precision-Recall Curve", fontsize=14, fontweight="bold"
    )
    ax.legend(loc="lower left")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

#### Model Comparison

##### Table

In [ ]:
def plot_model_comparison_table(metrics_list):

    df = pd.DataFrame(metrics_list)
    df = df.round(4)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.axis("tight")
    ax.axis("off")

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center",
        colWidths=[0.15] * len(df.columns),
    )

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)

    # Style header row
    for i, key in enumerate(df.columns):
        table[(0, i)].set_facecolor("#2E86AB")
        table[(0, i)].set_text_props(weight="bold", color="white")

    # Style alternating rows
    for i in range(1, len(df) + 1):
        for j in range(len(df.columns)):
            table[(i, j)].set_facecolor("#f8f9fa" if i % 2 == 0 else "#e9ecef")

    ax.set_title("Model Performance Comparison", fontsize=14, fontweight="bold", pad=20)

    plt.tight_layout()
    plt.show()

    return df

##### Bar Chart

In [ ]:
def plot_model_comparison_bars(metrics_list):

    df = pd.DataFrame(metrics_list)
    df.set_index("Model", inplace=True)

    metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    colors = ["#2E86AB", "#A23B72", "#F18F01", "#6C9A8F"]

    for i, metric in enumerate(metrics_to_plot):
        ax = axes[i]
        bars = ax.bar(
            df.index,
            df[metric],
            color=colors[: len(df.index)],
            edgecolor="black",
            linewidth=1.5,
        )
        ax.set_ylabel("Score", fontsize=11)
        ax.set_title(metric, fontsize=13, fontweight="bold")
        ax.set_ylim([0, 1])
        ax.grid(axis="y", alpha=0.3)

        # Add value labels
        for bar, val in zip(bars, df[metric]):
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                height + 0.02,
                f"{val:.3f}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold",
            )

    axes[-1].axis("off")  # Hide unused subplot

    fig.suptitle(
        "Model Performance Comparison Across All Metrics",
        fontsize=16,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

##### ROC Curve

In [ ]:
def plot_roc_comparison(y_true, y_probs_dict):

    fig, ax = plt.subplots(figsize=(10, 8))

    colors = ["#2E86AB", "#A23B72", "#F18F01", "#6C9A8F"]

    for i, (model_name, y_prob) in enumerate(y_probs_dict.items()):
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)
        ax.plot(
            fpr,
            tpr,
            color=colors[i],
            lw=2.5,
            label=f"{model_name} (AUC = {roc_auc:.3f})",
        )

    # Diagonal line (random classifier)
    ax.plot(
        [0, 1], [0, 1], color="#A23B72", lw=2, linestyle="--", label="Random Classifier"
    )

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title("ROC Curves - All Models Comparison", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

#### Training History Comparison

In [ ]:
def plot_training_comparison(histories_dict):

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    colors = ["#2E86AB", "#A23B72", "#F18F01", "#6C9A8F"]

    # Validation accuracy comparison
    for i, (model_name, history) in enumerate(histories_dict.items()):
        epochs = range(1, len(history.history["accuracy"]) + 1)
        axes[0].plot(
            epochs,
            history.history["val_accuracy"],
            "o-",
            color=colors[i],
            linewidth=2,
            markersize=4,
            label=model_name,
        )

    axes[0].set_xlabel("Epochs", fontsize=12)
    axes[0].set_ylabel("Validation Accuracy", fontsize=12)
    axes[0].set_title("Validation Accuracy Comparison", fontsize=14, fontweight="bold")
    axes[0].legend(loc="lower right")
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0, 1])

    # Validation loss comparison
    for i, (model_name, history) in enumerate(histories_dict.items()):
        epochs = range(1, len(history.history["loss"]) + 1)
        axes[1].plot(
            epochs,
            history.history["val_loss"],
            "o-",
            color=colors[i],
            linewidth=2,
            markersize=4,
            label=model_name,
        )

    axes[1].set_xlabel("Epochs", fontsize=12)
    axes[1].set_ylabel("Validation Loss", fontsize=12)
    axes[1].set_title("Validation Loss Comparison", fontsize=14, fontweight="bold")
    axes[1].legend(loc="upper right")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

#### Best Model Selection

In [ ]:
def select_best_model(metrics_list):

    df = pd.DataFrame(metrics_list)
    df.set_index("Model", inplace=True)

    # Calculate average score across main metrics
    df["Avg_Score"] = df[["Accuracy", "Precision", "Recall", "F1-Score"]].mean(axis=1)
    best_model = df["Avg_Score"].idxmax()
    best_score = df.loc[best_model, "Avg_Score"]

    # Create visualization
    fig, ax = plt.subplots(figsize=(10, 6))

    bars = ax.bar(
        df.index,
        df["Avg_Score"],
        color=["#2E86AB" if m != best_model else "#F18F01" for m in df.index],
        edgecolor="black",
        linewidth=1.5,
    )

    ax.set_ylabel("Average Performance Score", fontsize=12)
    ax.set_title("Model Performance Summary", fontsize=14, fontweight="bold")
    ax.set_ylim([0, 1])
    ax.grid(axis="y", alpha=0.3)

    # Add value labels
    for bar, val in zip(bars, df["Avg_Score"]):
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            height + 0.02,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )

    # Add recommendation text
    ax.text(
        0.5,
        -0.15,
        f"🏆 Best Model: {best_model} (Average Score: {best_score:.3f})",
        transform=ax.transAxes,
        ha="center",
        va="center",
        fontsize=13,
        fontweight="bold",
        bbox=dict(boxstyle="round", facecolor="#F18F01", alpha=0.3),
    )

    plt.tight_layout()
    plt.show()

    # Print detailed results
    print(f"\n{'=' * 60}")
    print(f"🏆 RECOMMENDATION: {best_model} is the best performing model")
    print(f"{'=' * 60}")
    print(f"\n📊 Key Metrics for {best_model}:")
    print(f"   • Accuracy:  {df.loc[best_model, 'Accuracy']:.4f}")
    print(f"   • Precision: {df.loc[best_model, 'Precision']:.4f}")
    print(f"   • Recall:    {df.loc[best_model, 'Recall']:.4f}")
    print(f"   • F1-Score:  {df.loc[best_model, 'F1-Score']:.4f}")
    print(f"   • ROC-AUC:   {df.loc[best_model, 'ROC-AUC']:.4f}")
    print(f"\n{'=' * 60}")

    return best_model, df

#### Train Models

##### EFFICIENTNETB0

In [ ]:
print("\n" + "=" * 60)
print("TRAINING: EfficientNetB0")
print("=" * 60)

model_effnet, history_effnet, y_true_effnet, y_pred_effnet, y_prob_effnet = (
    extract_features_and_train(
        "EfficientNetB0", train_generator, validation_generator, test_generator
    )
)

# Visualize results
plot_training_history(history_effnet, "EfficientNetB0")
plot_confusion_matrix(y_true_effnet, y_pred_effnet, "EfficientNetB0")
plot_roc_curve(y_true_effnet, y_prob_effnet, "EfficientNetB0")
plot_precision_recall_curve(y_true_effnet, y_prob_effnet, "EfficientNetB0")

# Store metrics
metrics_effnet = {
    "Model": "EfficientNetB0",
    "Accuracy": accuracy_score(y_true_effnet, y_pred_effnet),
    "Precision": precision_score(y_true_effnet, y_pred_effnet),
    "Recall": recall_score(y_true_effnet, y_pred_effnet),
    "F1-Score": f1_score(y_true_effnet, y_pred_effnet),
    "ROC-AUC": roc_auc_score(y_true_effnet, y_prob_effnet),
}

##### RESNET50

In [ ]:
print("\n" + "=" * 60)
print("TRAINING: ResNet50")
print("=" * 60)

model_resnet, history_resnet, y_true_resnet, y_pred_resnet, y_prob_resnet = (
    extract_features_and_train(
        "ResNet50", train_generator, validation_generator, test_generator
    )
)

# Visualize results
plot_training_history(history_resnet, "ResNet50")
plot_confusion_matrix(y_true_resnet, y_pred_resnet, "ResNet50")
plot_roc_curve(y_true_resnet, y_prob_resnet, "ResNet50")
plot_precision_recall_curve(y_true_resnet, y_prob_resnet, "ResNet50")

# Store metrics
metrics_resnet = {
    "Model": "ResNet50",
    "Accuracy": accuracy_score(y_true_resnet, y_pred_resnet),
    "Precision": precision_score(y_true_resnet, y_pred_resnet),
    "Recall": recall_score(y_true_resnet, y_pred_resnet),
    "F1-Score": f1_score(y_true_resnet, y_pred_resnet),
    "ROC-AUC": roc_auc_score(y_true_resnet, y_prob_resnet),
}

##### DENSENET201 (Optional)

In [ ]:
# Uncomment to train

# print("\n" + "=" * 60)
# print("TRAINING: DenseNet201")
# print("=" * 60)

# model_densenet, history_densenet, y_true_densenet, y_pred_densenet, y_prob_densenet = \
#     extract_features_and_train("DenseNet201", train_generator, validation_generator, test_generator)

# plot_training_history(history_densenet, "DenseNet201")
# plot_confusion_matrix(y_true_densenet, y_pred_densenet, "DenseNet201")
# plot_roc_curve(y_true_densenet, y_prob_densenet, "DenseNet201")

# metrics_densenet = {
#     "Model": "DenseNet201",
#     "Accuracy": accuracy_score(y_true_densenet, y_pred_densenet),
#     "Precision": precision_score(y_true_densenet, y_pred_densenet),
#     "Recall": recall_score(y_true_densenet, y_pred_densenet),
#     "F1-Score": f1_score(y_true_densenet, y_pred_densenet),
#     "ROC-AUC": roc_auc_score(y_true_densenet, y_prob_densenet),
# }

##### VGG16 (Optional)

In [ ]:
# Uncomment to train

# print("\n" + "=" * 60)
# print("TRAINING: VGG16")
# print("=" * 60)

# model_vgg16, history_vgg16, y_true_vgg16, y_pred_vgg16, y_prob_vgg16 = \
#     extract_features_and_train("VGG16", train_generator, validation_generator, test_generator)

# plot_training_history(history_vgg16, "VGG16")
# plot_confusion_matrix(y_true_vgg16, y_pred_vgg16, "VGG16")
# plot_roc_curve(y_true_vgg16, y_prob_vgg16, "VGG16")

# metrics_vgg16 = {
#     "Model": "VGG16",
#     "Accuracy": accuracy_score(y_true_vgg16, y_pred_vgg16),
#     "Precision": precision_score(y_true_vgg16, y_pred_vgg16),
#     "Recall": recall_score(y_true_vgg16, y_pred_vgg16),
#     "F1-Score": f1_score(y_true_vgg16, y_pred_vgg16),
#     "ROC-AUC": roc_auc_score(y_true_vgg16, y_prob_vgg16),
# }

#### Comprehensive Model Comparison

In [ ]:
# Collect metrics from trained models
metrics_list = [metrics_effnet, metrics_resnet]

# Add optional models if they were trained
# if 'metrics_densenet' in locals(): metrics_list.append(metrics_densenet)
# if 'metrics_vgg16' in locals(): metrics_list.append(metrics_vgg16)

print("\n" + "=" * 60)
print("📊 COMPREHENSIVE MODEL COMPARISON")
print("=" * 60)

# Display comparison table
comparison_df = plot_model_comparison_table(metrics_list)

# Plot bar chart comparison
plot_model_comparison_bars(metrics_list)

# Collect predictions for ROC comparison
y_probs_dict = {
    "EfficientNetB0": y_prob_effnet.flatten(),
    "ResNet50": y_prob_resnet.flatten(),
}
# if 'y_prob_densenet' in locals(): y_probs_dict["DenseNet201"] = y_prob_densenet.flatten()
# if 'y_prob_vgg16' in locals(): y_probs_dict["VGG16"] = y_prob_vgg16.flatten()

# Compare ROC curves
plot_roc_comparison(y_true_effnet, y_probs_dict)

# Compare training histories
histories_dict = {
    "EfficientNetB0": history_effnet,
    "ResNet50": history_resnet,
}
# if 'history_densenet' in locals(): histories_dict["DenseNet201"] = history_densenet
# if 'history_vgg16' in locals(): histories_dict["VGG16"] = history_vgg16

plot_training_comparison(histories_dict)

# Select best model
best_model, best_df = select_best_model(metrics_list)

#### Summary

In [ ]:
print("\n" + "=" * 70)
print(" COMPARATIVE EVALUATION OF DEEP LEARNING MODELS")
print(" FOR BRAIN STROKE DETECTION")
print("=" * 70)

print("\n📁 DATASET SUMMARY:")
print(f"   • Training images:   {train_generator.samples}")
print(f"   • Validation images: {validation_generator.samples}")
print(f"   • Testing images:    {test_generator.samples}")
print(f"   • Classes:           {list(train_generator.class_indices.keys())}")

print("\n🤖 MODELS EVALUATED:")
for m in metrics_list:
    print(f"   • {m['Model']}")

print("\n📊 PERFORMANCE SUMMARY:")
print("-" * 50)
for m in metrics_list:
    print(f"\n{m['Model']}:")
    print(f"   Accuracy:  {m['Accuracy']:.4f} ({m['Accuracy'] * 100:.2f}%)")
    print(f"   Precision: {m['Precision']:.4f}")
    print(f"   Recall:    {m['Recall']:.4f}")
    print(f"   F1-Score:  {m['F1-Score']:.4f}")
    print(f"   ROC-AUC:   {m['ROC-AUC']:.4f}")

print("\n🏆 BEST MODEL:")
print(f"   • {best_model}")
print(
    f"   • Accuracy:  {best_df.loc[best_model, 'Accuracy']:.4f} ({best_df.loc[best_model, 'Accuracy'] * 100:.2f}%)"
)
print(f"   • F1-Score:  {best_df.loc[best_model, 'F1-Score']:.4f}")
print(f"   • ROC-AUC:   {best_df.loc[best_model, 'ROC-AUC']:.4f}")

print("\n🔧 METHODOLOGY:")
print("   • Feature extraction using pre-trained ImageNet weights")
print("   • Lightweight classification head training")
print("   • Binary classification: Normal vs Stroke")

print("\n" + "=" * 70)
print("✅ ANALYSIS COMPLETE!")
print("=" * 70)